# Bing SERP Scraper (SeleniumBase)
Scrape Bing search-result URLs for a list of keywords, right here in Colab - no setup on your PC.

**Steps:** fill in keywords in Step 2, then Runtime > Run all, then download the `.txt` / `.csv` at the end.


In [ ]:
# Step 1 - install dependencies + SeleniumBase's managed Chrome (once per session)
!pip -q install seleniumbase
!sbase install chromedriver
!seleniumbase install chrome
print('browser setup done - if this is a fresh session, RESTART it now:')
print('Runtime > Restart session, then run all cells again (skip this cell next time).')


In [ ]:
# Step 2 - PUT YOUR KEYWORDS HERE (one per line)
KEYWORDS_RAW = """
python selenium tutorial
best coffee Bristol
"""
MAX_PAGES = 2  # pages per keyword
KEYWORDS = [k.strip() for k in KEYWORDS_RAW.splitlines() if k.strip()]
print(f'{len(KEYWORDS)} keyword(s), {MAX_PAGES} page(s) each:')
for k in KEYWORDS:
    print(' -', k)


In [ ]:
# Step 3 - scraper engine (headless for Colab)
import base64, time, datetime, urllib.parse
from selenium.webdriver.common.by import By
from selenium.common.exceptions import WebDriverException
from seleniumbase import Driver
baseurl = 'https://www.bing.com/search?q=%s&FORM=PERE&setLang=en'
blocked = ['microsoft.com','semrush.com','doubleclick.net','msclkid=','utm_campaign=','msockid=']
driver = None
def Log(t): print(datetime.datetime.now().strftime('%H:%M:%S') + ': ' + t)
def StartDriver():
    global driver
    ua = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/126.0.0.0 Safari/537.36'
    Log('Starting headless Chrome')
    # FIX: let SeleniumBase manage its own Chrome+driver (matching versions).
    # Do NOT pass binary_location (apt chromium mismatches SB's chromedriver).
    # --no-sandbox + --disable-dev-shm-usage are required in Colab's container.
    driver = Driver(uc=False, headless=True, agent=ua, window_size='1920,1080',
                    incognito=True, chromium_arg='no-sandbox,disable-dev-shm-usage,disable-gpu')
    Log('Driver started: ' + str(driver.capabilities.get('browserVersion', '?')))
def StopDriver():
    global driver
    try:
        if driver is not None: driver.quit()
    finally: driver = None
def ClickCookiePopup():
    for xp in ("//button[.//text()='Accept all']", '//div[@id="bnp_btn_accept"]'):
        try: driver.find_element(By.XPATH, xp).click(); break
        except Exception: pass
def GetPage(url, timeout=30):
    try:
        driver.set_page_load_timeout(timeout); driver.get(url)
    except WebDriverException: return 'ERROR: HOST_NOT_FOUND'
    ClickCookiePopup(); return driver.page_source


In [ ]:
# Step 3b - decode + scrape (continued)
def DecodeUrl(href):
    if not href: return None
    if 'bing.com/ck/a' not in href:
        return href if href.startswith('http') and all(d not in href for d in blocked) else None
    q = urllib.parse.parse_qs(urllib.parse.urlparse(href).query)
    u = q.get('u', [None])[0]
    if not u or not u.startswith('a1'): return None
    u = u[2:]; u += '=' * (-len(u) % 4)
    try: d = base64.urlsafe_b64decode(u).decode('utf-8')
    except Exception: return None
    return d if d.startswith('http') and all(x not in d for x in blocked) else None
XP = ['//a[@target="_blank" and contains(@h,"ID=SERP")]','//ol[@id="b_results"]//h2/a',
    '//ol[@id="b_results"]//a[contains(@href,"bing.com/ck/a")]']
def scrape_page(keyword, pagenum=0):
    Log(f"Getting '{keyword}' page {pagenum}")
    els = []
    if pagenum > 0:
        try: before = {e.get_attribute('href') for e in driver.find_elements(By.XPATH, XP[0])}
        except Exception: before = set()
        try:
            el = driver.find_element(By.XPATH, '//a[contains(@class,"sb_pagN")]')
            driver.execute_script('arguments[0].click();', el)
        except Exception as e: Log(f'No next page: {e}'); return ''
        import time as _t
        _t.sleep(3)
    else:
        c = GetPage(baseurl % urllib.parse.quote_plus(keyword))
        if c.startswith('ERROR:'): return c
        import time as _t2; _t2.sleep(2); ClickCookiePopup()
    for xp in XP:
        els = driver.find_elements(By.XPATH, xp)
        if els: break
    found = []
    hrefs = []
    for el in els:
        try: hrefs.append(el.get_attribute('href'))
        except Exception: pass
    seen_now = set()
    for h in hrefs:
        u = DecodeUrl(h)
        if u and u not in found: found.append(u)
    Log(f'{len(found)} links retrieved'); return chr(10).join(found)
print('Engine ready.')


In [ ]:
# Step 4 - run the scrape + SAVE RESULTS HERE
import csv
OUTPUT_TXT = '/content/bing_results.txt'
OUTPUT_CSV = '/content/bing_results.csv'
StartDriver(); rows = []
try:
    for kw in KEYWORDS:
        print(f'===== {kw} =====')
        for page in range(MAX_PAGES):
            chunk = scrape_page(kw, pagenum=page)
            if not chunk or chunk.startswith('ERROR:'):
                print(f'  page {page}: done ({chunk!r})'); break
            urls = [u for u in chunk.splitlines() if u.strip()]
            print(f'  page {page}: {len(urls)} urls')
            rows.extend((kw, page, u) for u in urls)
finally: StopDriver()
seen = set(); uniq = []
for kw, pg, u in rows:
    [uniq.append((kw, pg, u)) for _ in [0] if u not in seen]
    seen.add(u)
open(OUTPUT_TXT, 'w', encoding='utf-8').write(chr(10).join(u for _, _, u in uniq))
w = csv.writer(open(OUTPUT_CSV, 'w', encoding='utf-8', newline=''))
w.writerow(['keyword', 'page', 'url']); w.writerows(uniq)
print(f'Saved {len(uniq)} unique URLs')


In [ ]:
# Step 5 - download files to your computer
from google.colab import files
files.download('/content/bing_results.txt')
files.download('/content/bing_results.csv')


If the files panel shows the outputs, right-click > Download also works.
